# HR Analytics with Job Postings Data

In [`eda.ipynb`](./eda.ipynb) we explored the data jobs dataset row-by-row: how many postings exist, what fields are missing, what skills are most common. In this notebook we put on a different hat. We act as a **People Analytics / HR Analytics team** at a fictional employer and ask the kinds of questions that drive real talent decisions.

## What is HR analytics?

HR analytics (also called *people analytics* or *workforce analytics*) is the discipline of using data to make better decisions about an organization's people. Modern HR teams routinely use external labor market data — exactly like our Luke Barousse dataset — for:

- **Competitive compensation benchmarking.** Are we paying our data scientists in   Chicago at the 50th percentile of the market, or the 25th? If our offers keep   getting declined, the data tells us why.
- **Talent scouting and market mapping.** Where is the supply of senior data   engineers concentrated? Which competitors hire the most heavily, and what skills   are they screening for?
- **Retention and attrition risk.** When the external market for a skill heats up —   more postings, higher salaries, more remote options — internal employees with   that skill become flight risks.
- **Skill-gap planning.** What skills are growing fastest in postings? Should we   build training programs, or hire ahead of the curve?

We will walk through three of these workflows on the same dataset.

:::{important} Real-world caveats
Job postings are not a perfect mirror of the labor market. They overweight the minority of postings that disclose salary, they reflect *demand* rather than the actual workforce, and they include reposts. A real HR analytics team would join this data with internal HRIS records, surveys, and offer-acceptance data. The techniques shown here are still the same ones used in practice.
:::


In [ ]:
import polars as pl
import plotly.express as px

pl.Config.set_tbl_rows(20)
import plotly.io as pio
pio.templates.default = "simple_white"


## 📥 Load the Cleaned Dataset

We pick up where the EDA notebook left off, loading the preprocessed Parquet file with `job_skills` already parsed into a `List[str]` and `job_posted_date` parsed into a proper datetime.


In [ ]:
df = pl.read_parquet("data_jobs_preprocessed.parquet")
print("Rows:", df.height)
df.head(3)


## 🏢 Scenario

Imagine we work for **NorthStar Insights**, a mid-sized U.S. consulting firm. Leadership is planning the next fiscal year and the head of People asks three questions:

1. *Are our salary bands competitive for Data Analyst, Data Scientist, and Data Engineer roles?*
2. *Where should we focus our recruiting?*
3. *Which skills are heating up in the market — and which of our employees are most    exposed to being poached?*

Each section below tackles one of those questions.


## 1. Competitive Compensation Benchmarking

When People teams talk about "benchmarking", they almost always mean computing **percentiles** of the disclosed market salary for a given role and location, then comparing those percentiles to the company's own salary bands.

A typical band looks like this:

- **P25** — the bottom of the market. If you target this percentile you are likely   hiring entry-level or accepting longer time-to-fill.
- **P50 (median)** — "market rate". A reasonable default target for most roles.
- **P75** — paying for top talent or scarce skills.
- **P90** — premium / executive band.


In [ ]:
FOCUS_ROLES = ["Data Analyst", "Data Scientist", "Data Engineer"]

comp_bench = (
    df.filter(
        pl.col("job_country") == "United States",
        pl.col("salary_year_avg").is_not_null(),
        pl.col("job_title_short").is_in(FOCUS_ROLES),
    )
    .group_by("job_title_short")
    .agg(
        n=pl.len(),
        p25=pl.col("salary_year_avg").quantile(0.25),
        p50=pl.col("salary_year_avg").quantile(0.50),
        p75=pl.col("salary_year_avg").quantile(0.75),
        p90=pl.col("salary_year_avg").quantile(0.90),
    )
    .sort("p50", descending=True)
)
comp_bench


### Comparing NorthStar's bands to the market

Suppose NorthStar's current pay bands (midpoints) are:

| Role           | NorthStar midpoint |
| -------------- | ------------------ |
| Data Analyst   | \$78,000 |
| Data Scientist | \$130,000 |
| Data Engineer  | \$120,000 |

We can join this small table against the market percentiles and show the gap.


In [ ]:
northstar_bands = pl.DataFrame(
    {
        "job_title_short": ["Data Analyst", "Data Scientist", "Data Engineer"],
        "northstar_midpoint": [78_000, 130_000, 120_000],
    }
)

gap = (
    comp_bench.join(northstar_bands, on="job_title_short")
    .with_columns(
        gap_vs_p50=(pl.col("northstar_midpoint") - pl.col("p50")).round(0),
        position_in_market=(
            (pl.col("northstar_midpoint") - pl.col("p25"))
            / (pl.col("p75") - pl.col("p25"))
        ).round(2),
    )
    .select(
        "job_title_short",
        "northstar_midpoint",
        "p25",
        "p50",
        "p75",
        "gap_vs_p50",
        "position_in_market",
    )
)
gap


:::{tip} How to read `position_in_market`
We've expressed the company midpoint as a fraction of the IQR (P25 → P75). A value of `0.0` means we are paying at P25, `0.5` means we are sitting at P50, and `1.0` means we are at P75. If a role's value is much below `0.5`, expect declined offers and longer time-to-fill.
:::


## 2. Talent Scouting & Market Mapping

When NorthStar opens a new Data Engineer role, two questions matter immediately:

1. **Where is the supply?** Which cities have the most active Data Engineer postings? Those are also where we will find candidates.
2. **Who are we competing with?** Which companies are posting the most for the same role?


### Where is the demand for Data Engineers?


In [ ]:
de_locations = (
    df.filter(
        pl.col("job_title_short") == "Data Engineer",
        pl.col("job_country") == "United States",
    )
    .group_by("job_location")
    .agg(n=pl.len())
    .sort("n", descending=True)
    .head(15)
)
de_locations


In [ ]:
fig = px.bar(
    de_locations.to_pandas(),
    x="n",
    y="job_location",
    orientation="h",
    title="Top 15 U.S. cities by Data Engineer posting volume",
    template="simple_white",
 height=600)
fig.update_yaxes(categoryorder="total ascending")
fig.show()


Posting density is a reasonable proxy for talent density: where employers compete hardest, candidates also tend to congregate. NorthStar should consider opening remote-first or hub-based roles in the top three cities here.


### Who are the most active competitors?


In [ ]:
top_de_employers = (
    df.filter(
        pl.col("job_title_short") == "Data Engineer",
        pl.col("job_country") == "United States",
    )
    .group_by("company_name")
    .agg(n=pl.len())
    .sort("n", descending=True)
    .head(15)
)
top_de_employers


These companies are NorthStar's primary competitors for Data Engineering talent — and also a high-quality target list for outbound recruiting (people leaving these companies are likely to be qualified for our roles).


### Remote vs on-site mix

If we insist on on-site, we shrink the addressable pool. The data tells us how much.


In [ ]:
remote_mix = (
    df.filter(
        pl.col("job_title_short").is_in(FOCUS_ROLES),
        pl.col("job_country") == "United States",
    )
    .group_by("job_title_short")
    .agg(
        n=pl.len(),
        remote_share=pl.col("job_work_from_home").mean().round(3),
    )
    .sort("job_title_short")
)
remote_mix


## 3. Skill-Demand Heatmap & Retention Risk

When the *external* demand for a skill heats up, *internal* employees with that skill quietly become flight risks: their LinkedIn inboxes fill up, recruiters call them more often, and competing offers get more aggressive.

We can quantify this by tracking skill mention volume month over month. A simple year-over-year ratio is enough to surface skills with rising demand.


In [ ]:
skill_by_month = (
    df.filter(pl.col("job_skills").is_not_null())
    .with_columns(pl.col("job_posted_date").dt.truncate("1mo").alias("month"))
    .select("month", "job_skills")
    .explode("job_skills")
    .group_by(["month", "job_skills"])
    .agg(n=pl.len())
    .sort(["job_skills", "month"])
)
skill_by_month.head(10)


### Compare H1 of the most recent year to the previous year

We compute, for each skill, the share of postings mentioning it in the latest 6 months versus the prior 6 months. Skills with a sharp positive change are the ones the market is bidding up.


In [ ]:
from datetime import timedelta

max_month = df.select(pl.col("job_posted_date").max()).item()
cutoff_recent = max_month - timedelta(days=180)  # ~6 months back
cutoff_prior = max_month - timedelta(days=360)   # ~12 months back

def skill_share(start, end):
    window = df.filter(
        pl.col("job_skills").is_not_null(),
        pl.col("job_posted_date") >= start,
        pl.col("job_posted_date") < end,
    )
    total = window.height
    return (
        window.select("job_skills")
        .explode("job_skills")
        .group_by("job_skills")
        .agg(n=pl.len())
        .with_columns(share=(pl.col("n") / total))
    )

recent = skill_share(cutoff_recent, max_month).rename({"n": "n_recent", "share": "share_recent"})
prior = skill_share(cutoff_prior, cutoff_recent).rename({"n": "n_prior", "share": "share_prior"})

trend = (
    recent.join(prior, on="job_skills", how="inner")
    .filter((pl.col("n_recent") >= 200) & (pl.col("n_prior") >= 200))
    .with_columns(
        change_pct=((pl.col("share_recent") - pl.col("share_prior")) / pl.col("share_prior") * 100).round(1)
    )
    .sort("change_pct", descending=True)
    .select("job_skills", "share_prior", "share_recent", "change_pct")
)
trend.head(15)


### Translating the heatmap into retention actions

Once we know which skills are heating up, the People team can:

1. Pull the internal employee list and flag every employee whose primary skill    appears in the top 15 above. These are the **highest-risk retention targets**.
2. Prioritise *stay interviews* and proactive compensation reviews for those employees.
3. Build a 12-month skills-development roadmap so that *internal mobility* — not    external hiring — becomes the first line of defence.

:::{seealso} What if a skill is *cooling off*?
Skills with strongly negative change in the table above are also strategically interesting: they signal where market demand is waning and where the company should slow down hiring. They are also good candidates for re-skilling investments.
:::


## 📌 Recap

We took a single labor-market dataset and used it to mimic three distinct HR analytics workflows:

| Workflow | Question answered | Tools used |
| --- | --- | --- |
| Compensation benchmarking | *Are our salary bands competitive?* | quantile aggregations, joins to internal data |
| Talent scouting / market mapping | *Where do we hire and who do we compete with?* | group-by, value counts |
| Skill demand & retention risk | *Which skills are heating up — and which employees are flight risks?* | time-windowed `explode` + group-by |

The same dataset — nothing new ingested — supports all three. That is the recurring pattern in real HR analytics: pick a clean source of external market signal, and use it as a lens for compensation, sourcing, and retention decisions.
